# ROGII Rerun Noise and Micro-Tuning Filter

This notebook is a submission-quota decision aid for the ROGII code competition. It combines three forms of evidence before an agent spends one of the five daily submissions:

1. repeated leaderboard runs of byte-identical code,
2. an exact mathematical bound for the public model-package gate, and
3. RMS distances between audited public candidate outputs.

The central rule is simple: a new filename, parameter label, or upstream score is not an independent experiment. A candidate should earn a submission through a material prediction change, independent validation, or a genuinely different hidden execution lineage.

Measured scores from our 2026-07-18 audit were `9.571`, `7.123`, `7.010`, `7.170`, and `7.130`. The best measured route was A04 residual transfer at **7.010** (submission `54809311`).

## 1. Estimate the rerun-noise scale

The public ablation by `starsdaisuki/rogii-7-2x-pipeline-ablation` reports three submissions of the same kernel version: `7.216`, `7.246`, and `7.273`. These runs isolate stochastic execution noise from source-code changes.

In [ ]:
import math
import statistics

identical_kernel_scores = [7.216, 7.246, 7.273]
rerun_mean = statistics.mean(identical_kernel_scores)
rerun_sigma = statistics.stdev(identical_kernel_scores)
difference_sigma = math.sqrt(2.0) * rerun_sigma

print(f"identical-kernel mean: {rerun_mean:.4f}")
print(f"single-run sample sigma: {rerun_sigma:.4f}")
print(f"two-run difference sigma: {difference_sigma:.4f}")
print(f"conservative 2-sigma change threshold: {2*difference_sigma:.4f}")

A single score change below roughly two difference standard deviations is weak evidence. It may still be real, but it should not be described as an improvement without replication or a stronger local argument.

## 2. Bound the model-package gate before running it

The public disagreement gate has the form

$$g(d)=\frac{g_{max}}{1+(d/s)^2}, \qquad \Delta(d)=g(d)d.$$

For nonnegative disagreement $d$, the maximum possible row move is exactly $g_{max}s/2$, attained at $d=s$. Because an L2 norm is Lipschitz, the absolute RMSE change between two prediction vectors cannot exceed their maximum row move. This lets us reject some parameter sweeps without using Kaggle.

In [ ]:
def maximum_gated_row_move(gmax: float, scale: float) -> float:
    return gmax * scale / 2.0

scale = 6.0
for gmax in [0.005, 0.0075, 0.010, 0.015]:
    bound = maximum_gated_row_move(gmax, scale)
    print(f"gmax={gmax:0.4f}: max row move={bound:0.4f} ft")

step_bound = maximum_gated_row_move(0.010 - 0.005, scale)
print(f"one 0.005 gmax step can change any row by at most {step_bound:.4f} ft")

## 3. Compare prediction distance, not notebook names

The following distances were computed only after exact row-count and sample-ID-order validation. They are not leaderboard scores. They measure whether two candidate files are numerically different enough to represent a new route.

In [ ]:
audited_rms_distances_ft = {
    "model package 0.0075 vs 0.010": 0.005488,
    "model package 0.0075 vs Cycle8": 0.016465,
    "model package 0.010 vs Cycle8": 0.021953,
    "A04 vs MHA140 exact": 0.002734,
    "A04 vs WellBias Prefix RF": 0.371033,
    "A04 vs Cycle8": 0.397730,
    "A04 vs grouped OOF meta": 3.372194,
    "A04 vs LGB v40 adaptive": 5.127625,
}

def route_class(distance: float) -> str:
    if distance < 0.03:
        return "micro-variant: reject unless lineage evidence is exceptional"
    if distance < 0.15:
        return "small variant: require strong independent validation"
    return "materially different: eligible for full audit"

for pair, distance in audited_rms_distances_ft.items():
    print(f"{distance:8.6f} ft | {route_class(distance):62s} | {pair}")

Prediction distance is a triage signal, not a quality metric. A large distance can mean a useful independent model or a catastrophic failure. It only says that the candidate deserves a full validation audit rather than immediate duplicate rejection.

## 4. Agent decision protocol

For each candidate, record:

- source notebook, author, version, and claimed score;
- exact dependency graph and whether any fallback ran;
- grouped OOF or visible-prefix validation evidence;
- final `submission.csv` row count, ID order, finite check, and SHA-256;
- RMS distance to the current best and to other queued candidates;
- daily submission count before and after the action.

Then apply this order:

1. Reject malformed, duplicate, or unlicensed artifacts.
2. Reject changes resolved by a mathematical bound or exact local comparison.
3. Run the candidate privately on Kaggle and audit its final output.
4. Submit only the audited private version.
5. After the real score arrives, publish a separate documentation version with the score and reference. Never submit that documentation rerun.

This workflow does not eliminate leaderboard noise. It makes the source of uncertainty explicit and preserves scarce submissions for experiments that can teach us something.

## Attribution and reproducibility notes

- Rerun-noise and gate-bound evidence: `starsdaisuki/rogii-7-2x-pipeline-ablation`.
- Public pipeline lineage discussed there includes work by bernubritz, degnonguidi, baidalinadilzhan, fleongg, sp45, iaztec, pilkwang, yusuketogashi, kevinermille, and arksey.
- Candidate distances and the five measured 2026-07-18 scores come from our own exact-order output audits and Kaggle submissions.
- No competition target data is read by this notebook; every numeric input is declared above, so the analysis is deterministic and easy to fork.